# 3.1.6, Courant algebroid — five axioms by definitions

**Goal.** Drive each of the five Courant-algebroid axioms
(D-compatibility, anchor compatibility, Vaisman Leibniz,
inner-product compatibility, cyclic Jacobi) on the standard exact
Courant algebroid $\mathrm{TM}\oplus T^*M$ through fully
definitional `ProofChain`s — every step tagged
`provenance_tag="axiom"`, no seeded-theorem citation shortcuts.

We use the same algebroid wrapper that backs
`prove_jacobi_reduction` and `prove_courant_dorfman_bridge`
(both seeded-theorem versions). The Stage E methods
exercised here — `prove_D_compat`, `prove_anchor_compat`,
`prove_leibniz`, `prove_inner_compat`,
`prove_jacobi_by_definitions` — emit one or more atomic
definitional rewrites per axiom and never collapse the chain
into a single citation step.

## Strategy

Each axiom is proved on generic operands — section pairs
$e_1=(X,\alpha)$, $e_2=(Y,\beta)$, $e_3=(Z,\gamma)$ and a scalar
function $f$ — with no concrete frame substituted in. The chain's
first cell builds the symbolic operands and a `PropertyRegistry`
that declares each component's degree (vector half degree 0, form
half degree 1).

For each axiom we:

1. Build the LHS Expr through the public Stage E API
   (`C.D(f)`, `C.anchor_of(e1)`, `C.inner_product(e1, e2)`,
   `BracketApply(C.courant, e1, e2)`).
2. Call the corresponding `prove_*` method to obtain a
   `ProofChain`.
3. Print every step's `rule` / `provenance_tag` /
   `justification`, plus the final Expr.
4. Assert the chain's structural invariants (length, all-axiom,
   step-by-step consistency).

Section 6 collects the chain lengths and rule signatures into a
summary table.

In [1]:
# Make jacopy importable when the notebook is opened directly.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401


## 1. Setup

**Sections.** $e_1=(X,\alpha)$, $e_2=(Y,\beta)$, $e_3=(Z,\gamma)$.
**Scalar.** $f \in C^\infty(M)$. **Algebroid.** Standard exact
$\mathrm{TM}\oplus T^*M$, no $H$-twist (the prove methods
themselves handle the twisted case symmetrically — see the
Section 5 commentary).

In [2]:
from jacopy.brackets.dorfman import SectionPair
from jacopy.core.expr import Symbol
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.library.courant_algebroid import CourantAlgebroid

C = CourantAlgebroid()

X, Y, Z = Symbol("X"), Symbol("Y"), Symbol("Z")
alpha, beta, gamma = Symbol("α"), Symbol("β"), Symbol("γ")
f = Symbol("f")

registry = PropertyRegistry()
for sym, deg in [(X, 0), (Y, 0), (Z, 0), (alpha, 1), (beta, 1), (gamma, 1), (f, 0)]:
    registry.declare(sym, Graded(degree=deg))

e1 = SectionPair(X, alpha)
e2 = SectionPair(Y, beta)
e3 = SectionPair(Z, gamma)

print(f"Algebroid: {C.name}")
print(f"e1 = {e1._repr_inner()}")
print(f"e2 = {e2._repr_inner()}")
print(f"e3 = {e3._repr_inner()}")
print(f"f  = {f._repr_inner()}")


Algebroid: Courant(TM⊕T*M)
e1 = (X, α)
e2 = (Y, β)
e3 = (Z, γ)
f  = f


In [3]:
def show_chain(name, chain):
    """Print every step of a ProofChain plus its terminal forms."""
    print(f"=== {name}: {len(chain)} steps ===")
    print(f"Initial: {chain.initial._repr_inner()}")
    for i, step in enumerate(chain.steps, 1):
        tag = step.provenance_tag or "-"
        print(f"  Step {i}: rule={step.rule} [tag={tag}]")
        print(f"    {step.before._repr_inner()[:90]}")
        print(f"    → {step.after._repr_inner()[:90]}")
    print(f"Final:   {chain.final._repr_inner()}")

def chain_invariants(chain):
    """Sanity checks that hold for every Stage E chain."""
    assert len(chain) >= 1, "chain is empty"
    for step in chain.steps:
        assert step.provenance_tag == "axiom", (
            f"non-axiom step: {step.rule} tagged {step.provenance_tag}"
        )
    for i in range(1, len(chain)):
        assert chain.steps[i].before == chain.steps[i - 1].after, (
            f"chain consistency broken at step {i}: "
            f"step{i}.before != step{i-1}.after"
        )


## 2. D-compatibility, $\mathrm{anchor}(D f) = 0$

On the standard exact algebroid $D f := (0, df)$, so the anchor
(canonical projection $\pi_{TM}$) returns the vector half $0$.
Two atomic axiom rewrites: $D$-operator definition + anchor
projection.

In [4]:
chain_D = C.prove_D_compat(f)
show_chain("prove_D_compat", chain_D)
chain_invariants(chain_D)
assert len(chain_D) == 2
assert chain_D.final._repr_inner() == "0"
print("\n✓ D-compatibility: 2 axiom steps, final = 0.")


=== prove_D_compat: 2 steps ===
Initial: anchor(D(f))
  Step 1: rule=DOperatorDefinition [tag=axiom]
    anchor(D(f))
    → anchor((0, d(f)))
  Step 2: rule=CourantAnchorDefinition [tag=axiom]
    anchor((0, d(f)))
    → 0
Final:   0

✓ D-compatibility: 2 axiom steps, final = 0.


## 3. Anchor compatibility, $\mathrm{anchor}([e_1, e_2]_C) = [\mathrm{anchor}(e_1), \mathrm{anchor}(e_2)]_{VF}$

Two atomic axiom rewrites: Courant bracket definition (form half
is irrelevant) + anchor projection. The chain's final form is
the inert vector-bracket apply $[X, Y]_{VF}$, which equals
$[\mathrm{anchor}(e_1), \mathrm{anchor}(e_2)]_{VF}$ once anchor
projection collapses each side.

In [5]:
chain_A = C.prove_anchor_compat(e1, e2, registry=registry)
show_chain("prove_anchor_compat", chain_A)
chain_invariants(chain_A)
assert len(chain_A) == 2
print("\n✓ Anchor-compatibility: 2 axiom steps, final = vector-bracket apply.")


=== prove_anchor_compat: 2 steps ===
Initial: anchor([·,·]_C((X, α), (Y, β)))
  Step 1: rule=CourantBracketDefinition [tag=axiom]
    anchor([·,·]_C((X, α), (Y, β)))
    → anchor(([·,·](X, Y), (L_X(β) + (-L_Y(α)) + (-(1/2 * d((ι_X(β) + (-ι_Y(α)))))))))
  Step 2: rule=CourantAnchorDefinition [tag=axiom]
    anchor(([·,·](X, Y), (L_X(β) + (-L_Y(α)) + (-(1/2 * d((ι_X(β) + (-ι_Y(α)))))))))
    → [·,·](X, Y)
Final:   [·,·](X, Y)

✓ Anchor-compatibility: 2 axiom steps, final = vector-bracket apply.


## 4. Vaisman Leibniz, $[e_1, f e_2]_C = f [e_1, e_2]_C + (\mathrm{anchor}(e_1)\,f) e_2 - \langle e_1, e_2 \rangle\,D f$

The granular **8-step** chain:

1. `CourantBracketDefinition` — unfold Courant on $f$-scaled
   second argument.
2. `LieBracketLeibnizSecondSlot` — $[X, fY] = f[X,Y] + X(f) Y$.
3. `LieDerivativeProductRule` — $\mathcal{L}_X(f\beta) = f\,\mathcal{L}_X\beta + X(f)\,\beta$.
4. `LieRescaling` — $\mathcal{L}_{fY}\alpha = f\,\mathcal{L}_Y\alpha + \alpha(Y) df$.
5. `InteriorScalarLinearity` — $\iota_{fV}\omega = f\,\iota_V\omega$ (×2 + twist).
6. `InteriorPairing` — $\iota_V\omega = \omega(V)$ on a 1-form (×2).
7. `ExteriorDProductRule` — $d(f g) = df\cdot g + f\,d g$.
8. `VaismanLeibnizRegroup` — collect by $f$, $X(f)$, $df$ coefficients.

In [6]:
chain_L = C.prove_leibniz(e1, e2, f, registry=registry)
show_chain("prove_leibniz", chain_L)
chain_invariants(chain_L)
assert len(chain_L) == 8
expected = [
    "CourantBracketDefinition",
    "LieBracketLeibnizSecondSlot",
    "LieDerivativeProductRule",
    "LieRescaling",
    "InteriorScalarLinearity",
    "InteriorPairing",
    "ExteriorDProductRule",
    "VaismanLeibnizRegroup",
]
assert [s.rule for s in chain_L.steps] == expected
print("\n✓ Vaisman Leibniz: 8 atomic axioms in expected order.")


=== prove_leibniz: 8 steps ===
Initial: [·,·]_C((X, α), ((f * Y), (f * β)))
  Step 1: rule=CourantBracketDefinition [tag=axiom]
    [·,·]_C((X, α), ((f * Y), (f * β)))
    → ([·,·](X, (f * Y)), (L_X((f * β)) + (-L_(f * Y)(α)) + (-(1/2 * d((ι_X((f * β)) + (-ι_(f * 
  Step 2: rule=LieBracketLeibnizSecondSlot [tag=axiom]
    ([·,·](X, (f * Y)), (L_X((f * β)) + (-L_(f * Y)(α)) + (-(1/2 * d((ι_X((f * β)) + (-ι_(f * 
    → (((f * [·,·](X, Y)) + (L_X(f) * Y)), (L_X((f * β)) + (-L_(f * Y)(α)) + (-(1/2 * d((ι_X((f 
  Step 3: rule=LieDerivativeProductRule [tag=axiom]
    (((f * [·,·](X, Y)) + (L_X(f) * Y)), (L_X((f * β)) + (-L_(f * Y)(α)) + (-(1/2 * d((ι_X((f 
    → (((f * [·,·](X, Y)) + (L_X(f) * Y)), ((f * L_X(β)) + (L_X(f) * β) + (-L_(f * Y)(α)) + (-(1
  Step 4: rule=LieRescaling [tag=axiom]
    (((f * [·,·](X, Y)) + (L_X(f) * Y)), ((f * L_X(β)) + (L_X(f) * β) + (-L_(f * Y)(α)) + (-(1
    → (((f * [·,·](X, Y)) + (L_X(f) * Y)), ((f * L_X(β)) + (L_X(f) * β) + (-(f * L_Y(α))) + (-(⟨
  Step 5: ru

## 5. Inner-product compatibility, $\rho(e_1) \langle e_2, e_3 \rangle = \langle [e_1, e_2]_C + D\langle e_1, e_2 \rangle, e_3 \rangle + \langle e_2, [e_1, e_3]_C + D\langle e_1, e_3 \rangle \rangle$

**7 atomic axiom steps.** The chain unfolds the LHS to a
canonical pairing form (steps 1-3), inserts the
$d\alpha$-antisymmetry zero combination (step 4), then refolds
**backward** through three definitional axioms — inner-product
definition, Dorfman bracket definition, Courant–Dorfman bridge —
to land on the Vaisman RHS.

In [7]:
chain_I = C.prove_inner_compat(e1, e2, e3, registry=registry)
show_chain("prove_inner_compat", chain_I)
chain_invariants(chain_I)
assert len(chain_I) == 7
expected = [
    "CourantInnerProductDefinition",
    "PairingLieLeibniz",
    "VectorLieDerivativeIsBracket",
    "DAlphaAntisymmetry",
    "CourantInnerProductDefinition",
    "DorfmanBracketDefinition",
    "CourantDorfmanBridge",
]
assert [s.rule for s in chain_I.steps] == expected
print("\n✓ Inner-product compat: 7 atomic axioms (LHS → canonical → RHS).")


=== prove_inner_compat: 7 steps ===
Initial: L_X(⟨(Y, β), (Z, γ)⟩)
  Step 1: rule=CourantInnerProductDefinition [tag=axiom]
    L_X(⟨(Y, β), (Z, γ)⟩)
    → L_X((1/2 * (⟨β, Z⟩ + ⟨γ, Y⟩)))
  Step 2: rule=PairingLieLeibniz [tag=axiom]
    L_X((1/2 * (⟨β, Z⟩ + ⟨γ, Y⟩)))
    → (1/2 * (⟨L_X(β), Z⟩ + ⟨β, L_X(Z)⟩ + ⟨L_X(γ), Y⟩ + ⟨γ, L_X(Y)⟩))
  Step 3: rule=VectorLieDerivativeIsBracket [tag=axiom]
    (1/2 * (⟨L_X(β), Z⟩ + ⟨β, L_X(Z)⟩ + ⟨L_X(γ), Y⟩ + ⟨γ, L_X(Y)⟩))
    → (1/2 * (⟨L_X(β), Z⟩ + ⟨β, [·,·](X, Z)⟩ + ⟨L_X(γ), Y⟩ + ⟨γ, [·,·](X, Y)⟩))
  Step 4: rule=DAlphaAntisymmetry [tag=axiom]
    (1/2 * (⟨L_X(β), Z⟩ + ⟨β, [·,·](X, Z)⟩ + ⟨L_X(γ), Y⟩ + ⟨γ, [·,·](X, Y)⟩))
    → ((1/2 * (⟨L_X(β), Z⟩ + ⟨β, [·,·](X, Z)⟩ + ⟨L_X(γ), Y⟩ + ⟨γ, [·,·](X, Y)⟩)) + (1/2 * ((-⟨ι_
  Step 5: rule=CourantInnerProductDefinition [tag=axiom]
    ((1/2 * (⟨L_X(β), Z⟩ + ⟨β, [·,·](X, Z)⟩ + ⟨L_X(γ), Y⟩ + ⟨γ, [·,·](X, Y)⟩)) + (1/2 * ((-⟨ι_
    → (⟨([·,·](X, Y), (L_X(β) + (-ι_Y(d(α))))), (Z, γ)⟩ + ⟨(Y, β), ([·,·](X, Z), (L_X(

## 6. Cyclic Jacobi by definitions

**4 atomic axiom steps**, alternative to `prove_jacobi_reduction`
(which uses a single seeded-theorem citation):

1. `CyclicCourantJacobiatorDefinition` — Sum of three nested outer
   brackets.
2. `CourantDorfmanBridge` ×3 — outer brackets become Dorfman + D-correction.
3. `CyclicDInnerProductCancellation` — D-correction cyclic sum cancels.
4. `DorfmanLodayClosure` — Dorfman Loday identity collapses the cyclic
   nested sum to the algebroid's `jacobi_condition` obstruction
   (zero in the untwisted case).

In [8]:
chain_J = C.prove_jacobi_by_definitions(e1, e2, e3)
show_chain("prove_jacobi_by_definitions", chain_J)
chain_invariants(chain_J)
assert len(chain_J) == 4
from jacopy.core.expr import Integer
assert chain_J.final == Integer(0), "untwisted Jacobi should land at 0"
print("\n✓ Cyclic Jacobi (untwisted): 4 atomic axioms, final = 0.")


=== prove_jacobi_by_definitions: 4 steps ===
Initial: ([·,·]_C([·,·]_C((X, α), (Y, β)), (Z, γ)) + [·,·]_C([·,·]_C((Y, β), (Z, γ)), (X, α)) + [·,·]_C([·,·]_C((Z, γ), (X, α)), (Y, β)))
  Step 1: rule=CyclicCourantJacobiatorDefinition [tag=axiom]
    ([·,·]_C([·,·]_C((X, α), (Y, β)), (Z, γ)) + [·,·]_C([·,·]_C((Y, β), (Z, γ)), (X, α)) + [·,
    → ([·,·]_C([·,·]_C((X, α), (Y, β)), (Z, γ)) + [·,·]_C([·,·]_C((Y, β), (Z, γ)), (X, α)) + [·,
  Step 2: rule=CourantDorfmanBridge [tag=axiom]
    ([·,·]_C([·,·]_C((X, α), (Y, β)), (Z, γ)) + [·,·]_C([·,·]_C((Y, β), (Z, γ)), (X, α)) + [·,
    → (([·,·]_D([·,·]_C((X, α), (Y, β)), (Z, γ)) + (-D(⟨[·,·]_C((X, α), (Y, β)), (Z, γ)⟩))) + ([
  Step 3: rule=CyclicDInnerProductCancellation [tag=axiom]
    (([·,·]_D([·,·]_C((X, α), (Y, β)), (Z, γ)) + (-D(⟨[·,·]_C((X, α), (Y, β)), (Z, γ)⟩))) + ([
    → ([·,·]_D([·,·]_C((X, α), (Y, β)), (Z, γ)) + [·,·]_D([·,·]_C((Y, β), (Z, γ)), (X, α)) + [·,
  Step 4: rule=DorfmanLodayClosure [tag=axiom]
    ([·,·]_D([·,·]_C((X, α

### H-twisted variant

The same 4-step chain on the $H$-twisted algebroid emits the
twist obstruction $d H$ as the final form — Jacobi closes
exactly when $d H = 0$.

In [9]:
H = Symbol("H")
C_twist = CourantAlgebroid(background_H=H)
registry.declare(H, Graded(degree=3))

chain_Jt = C_twist.prove_jacobi_by_definitions(e1, e2, e3)
show_chain("prove_jacobi_by_definitions (twisted)", chain_Jt)
chain_invariants(chain_Jt)
assert len(chain_Jt) == 4
assert "H" in repr(chain_Jt.final) and "d" in repr(chain_Jt.final)
print("\n✓ Cyclic Jacobi (twisted): 4 atomic axioms, final carries dH.")


=== prove_jacobi_by_definitions (twisted): 4 steps ===
Initial: ([·,·]_C([·,·]_C((X, α), (Y, β)), (Z, γ)) + [·,·]_C([·,·]_C((Y, β), (Z, γ)), (X, α)) + [·,·]_C([·,·]_C((Z, γ), (X, α)), (Y, β)))
  Step 1: rule=CyclicCourantJacobiatorDefinition [tag=axiom]
    ([·,·]_C([·,·]_C((X, α), (Y, β)), (Z, γ)) + [·,·]_C([·,·]_C((Y, β), (Z, γ)), (X, α)) + [·,
    → ([·,·]_C([·,·]_C((X, α), (Y, β)), (Z, γ)) + [·,·]_C([·,·]_C((Y, β), (Z, γ)), (X, α)) + [·,
  Step 2: rule=CourantDorfmanBridge [tag=axiom]
    ([·,·]_C([·,·]_C((X, α), (Y, β)), (Z, γ)) + [·,·]_C([·,·]_C((Y, β), (Z, γ)), (X, α)) + [·,
    → (([·,·]_D([·,·]_C((X, α), (Y, β)), (Z, γ)) + (-D(⟨[·,·]_C((X, α), (Y, β)), (Z, γ)⟩))) + ([
  Step 3: rule=CyclicDInnerProductCancellation [tag=axiom]
    (([·,·]_D([·,·]_C((X, α), (Y, β)), (Z, γ)) + (-D(⟨[·,·]_C((X, α), (Y, β)), (Z, γ)⟩))) + ([
    → ([·,·]_D([·,·]_C((X, α), (Y, β)), (Z, γ)) + [·,·]_D([·,·]_C((Y, β), (Z, γ)), (X, α)) + [·,
  Step 4: rule=DorfmanLodayClosure [tag=axiom]
    ([·,·]_D([·,

## 7. Summary

All five Courant-algebroid axioms close on $\mathrm{TM}\oplus T^*M$
via fully definitional `ProofChain`s — every step carries
`provenance_tag="axiom"`, no seeded-theorem citation shortcuts.

In [10]:
summary = [
    ("D-compatibility",          chain_D),
    ("Anchor-compatibility",     chain_A),
    ("Vaisman Leibniz",          chain_L),
    ("Inner-product compat",     chain_I),
    ("Cyclic Jacobi (untwisted)", chain_J),
    ("Cyclic Jacobi (twisted)",  chain_Jt),
]
print(f"{'Axiom':<32}{'Steps':>8}  All-axiom?  Final")
print("-" * 72)
for label, ch in summary:
    all_axiom = all(s.provenance_tag == "axiom" for s in ch.steps)
    final_repr = ch.final._repr_inner()
    if len(final_repr) > 24:
        final_repr = final_repr[:21] + "..."
    print(f"{label:<32}{len(ch):>8}  {str(all_axiom):>10}  {final_repr}")
print()
print("All chains: every step provenance_tag='axiom' (no seeded theorem citation).")


Axiom                              Steps  All-axiom?  Final
------------------------------------------------------------------------
D-compatibility                        2        True  0
Anchor-compatibility                   2        True  [·,·](X, Y)
Vaisman Leibniz                        8        True  (((f * [·,·](X, Y)) +...
Inner-product compat                   7        True  (⟨([·,·]_C((X, α), (Y...
Cyclic Jacobi (untwisted)              4        True  0
Cyclic Jacobi (twisted)                4        True  d(H)

All chains: every step provenance_tag='axiom' (no seeded theorem citation).
